In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.cluster import KMeans
import joblib
import plotly.graph_objects as go
import plotly.express as px 
import plotly.io as pio
import plotly.subplots as sp
from sklearn.pipeline import Pipeline 
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.Tenth_Best_Time_Estimator import TenthBestTimeEstimator


In [3]:
df= pd.read_parquet(repo_root / 'data' / 'processed' / 'reunion_segments_cleaned.parquet')

In [4]:
print(f"📊 Dataset: {len(df)} segments")
print(f"   Running: {(df['activity_type']=='Run').sum()}")
print(f"   Cycling: {(df['activity_type']=='Ride').sum()}")

📊 Dataset: 2848 segments
   Running: 1470
   Cycling: 1378


In [5]:
# Création de la figure avec deux sous-graphiques
fig = sp.make_subplots(rows=2, cols=1,
                    subplot_titles=("Distribution of Effort Counts", "Best Time vs Effort Count"),
                    vertical_spacing=0.15)

# 1. Histogramme (échelle log)
histogram = go.Histogram(
    x=df['total_effort_count'],
    nbinsx=50,
    marker=dict(color='blue', line=dict(color='black', width=1)),
    name='Effort Count'
)
fig.add_trace(histogram, row=1, col=1)

# Ligne de seuil (500)
fig.add_vline(x=500, line=dict(color='red', dash='dash'), row=1, col=1, annotation_text="Threshold (500)", annotation_position="top right")

# Configuration de l'échelle log pour l'axe y
fig.update_yaxes(type="log", row=1, col=1)
fig.update_xaxes(title_text="Total Efforts", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

# 2. Scatter plot (échelle log pour x, colorbar pour la distance)
scatter = go.Scatter(
    x=df['total_effort_count'],
    y=df['best_time'],
    mode='markers',
    marker=dict(
        size=8,
        color=df['distance']/1000,
        colorscale='Viridis',
        opacity=0.6,
        showscale=True,
        colorbar=dict(title='Distance (km)')
    ),
    name='Best Time'
)
fig.add_trace(scatter, row=2, col=1)

# Configuration des axes pour le scatter plot
fig.update_xaxes(type="log", title_text="Total Efforts", row=2, col=1)
fig.update_yaxes(title_text="Best Time (s)", row=2, col=1)

# Mise en page finale
fig.update_layout(
    height=800,
    width=800,
    showlegend=False,
    title_text="Effort Count Analysis"
)

# Affichage
fig.show()

In [ ]:
Train, Test = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
Train_ride = Train[Train['activity_type']=='Ride']

###Physic

In [7]:
def effort_correction_model(total_effort_count, best_time, asymptote_factor=0.95):
    """
    Model: theoretical_best = best_time * asymptote_factor * (1 - exp(-k * effort_count))
    
    As effort_count → ∞, multiplier → asymptote_factor
    As effort_count → 0, multiplier → 0 (big correction needed)
    """
    
    # Fit on segments with high effort counts (your "ground truth")
    # Assumption: segments with 1000+ efforts are well-sampled
    high_effort_mask = total_effort_count > 1000
    
    if high_effort_mask.sum() > 50:  # Need enough data
        # For high-effort segments, assume best_time is within 95% of theoretical
        # Use this to calibrate
        
        def saturation_curve(effort, k):
            return asymptote_factor * (1 - np.exp(-k * effort))
        
        # Fit k parameter
        high_effort_data = total_effort_count[high_effort_mask]
        # Target: we assume high effort segments have multiplier ≈ 0.95
        k_param = -np.log(1 - 0.95/asymptote_factor) / np.median(high_effort_data)
        
        # Apply to all segments
        correction_factor = saturation_curve(total_effort_count, k_param)
        theoretical_best = best_time * correction_factor
        
        return theoretical_best, correction_factor
    else:
        # Not enough high-effort segments, use simpler model
        return best_time * 0.95, np.ones_like(best_time) * 0.95

In [8]:
def visualize_effort_effect(df):
    # Normalize times by segment difficulty for fair comparison
    # Simple normalization: time per (km + 10m elevation)
    df['difficulty_units'] = df['distance']/1000 + df['elevation_gain']/10
    df['normalized_time'] = df['best_time'] / df['difficulty_units']
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Raw relationship
    axes[0].scatter(df['total_effort_count'], df['normalized_time'], alpha=0.5)
    axes[0].set_xlabel('Total Effort Count')
    axes[0].set_ylabel('Normalized Best Time')
    axes[0].set_xscale('log')
    axes[0].set_title('Effect of Sample Size on Best Time')
    
    # Plot 2: Percentile by effort bucket
    effort_bins = [0, 100, 500, 1000, 5000, np.inf]
    for i in range(len(effort_bins)-1):
        mask = (df['total_effort_count'] >= effort_bins[i]) & \
               (df['total_effort_count'] < effort_bins[i+1])
        if mask.sum() > 10:
            data = df[mask]['normalized_time']
            axes[1].boxplot([data], positions=[i], widths=0.6)
    
    axes[1].set_xlabel('Effort Count Bucket')
    axes[1].set_ylabel('Normalized Best Time')
    axes[1].set_xticks(range(len(effort_bins)-1))
    axes[1].set_xticklabels([f'{int(effort_bins[i])}-{int(effort_bins[i+1])}' 
                              for i in range(len(effort_bins)-1)])
    
    plt.tight_layout()
    plt.show()